# Benchmark: DRNN + Optuna (Direct Competitor)

**Author** : Anwesha Singh — CSE, Manipal University Jaipur

---

## Purpose

This notebook trains the **identical DRNN architecture** as the proposed model,  
but replaces NiOA with **Optuna** (Tree-structured Parzen Estimator, TPE)  
for hyperparameter optimisation.  

This is the most critical benchmark, as it directly isolates the contribution  
of the NiOA optimisation strategy from the architecture itself.

### Fairness constraints enforced
| Constraint | Value |
|---|---|
| Architecture | Identical to NiOA-DRNN |
| Search space | Identical |
| Optimisation subset | 30 % of training data |
| Trials budget | `N_AGENTS × (MAX_ITERATIONS + 1)` (equal function evaluations) |
| Time limit per trial | Same `OPT_TIME_LIMIT` seconds |
| Splits | Identical canonical frozen splits |
| Evaluation metrics | Identical |

The number of Optuna trials is set equal to the total number of objective  
function evaluations performed by NiOA:  
`N_AGENTS × (1 + MAX_ITERATIONS)` = `6 × 7 = 42`.


In [ ]:
import os, sys, json, gc, warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import tensorflow as tf
import tensorflow.keras.backend as K
from datetime import datetime
from tensorflow.keras.callbacks import EarlyStopping

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError:
    raise ImportError("Run:  pip install optuna")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

from core.config  import (
    SPLITS_DIR, BENCHMARK_DIR, NIOA_RESULTS_DIR,
    N_AGENTS, MAX_ITERATIONS, OPT_SUBSET_RATIO,
    OPT_EPOCHS, OPT_PATIENCE, OPT_TIME_LIMIT,
    FINAL_EPOCHS, FINAL_PATIENCE, SEQUENCE_LENGTH,
    RANDOM_SEED, FORECAST_HORIZONS, HYPERPARAMETER_BOUNDS,
)
from core.utils   import configure_gpu, set_seeds, make_tf_dataset, to_python_types
from core.models  import create_lstm_model
from core.train   import TimeLimitCallback
from core.evaluate import evaluate_keras_model
from benchmarking.utils.data_loader import load_splits
from benchmarking.utils.metrics     import save_benchmark_results, compute_metrics

configure_gpu()
set_seeds(RANDOM_SEED)
sns.set(style="whitegrid")
print(f"TensorFlow : {tf.__version__}")
print(f"Optuna     : {optuna.__version__}")

In [ ]:
# ============================================================
HORIZON = 60
# ============================================================

assert HORIZON in FORECAST_HORIZONS

X_train, y_train, X_val, y_val, X_test, y_test, scaler, meta = \
    load_splits(SPLITS_DIR, HORIZON)

SEQ_LEN   = X_train.shape[1]
NUM_FEATS = X_train.shape[2]

# Optimisation subset
N_OPT_TRAIN = int(len(X_train) * OPT_SUBSET_RATIO)
N_OPT_VAL   = int(len(X_val)   * OPT_SUBSET_RATIO)

X_tr_opt = X_train[:N_OPT_TRAIN]
y_tr_opt = y_train[:N_OPT_TRAIN]
X_va_opt = X_val[:N_OPT_VAL]
y_va_opt = y_val[:N_OPT_VAL]

# Equal function-evaluation budget to NiOA
# NiOA: N_AGENTS initial evaluations + N_AGENTS × MAX_ITERATIONS refinements
N_TRIALS = N_AGENTS * (1 + MAX_ITERATIONS)

print(f"Horizon k = {HORIZON} s")
print(f"seq_len={SEQ_LEN}  |  num_feats={NUM_FEATS}")
print(f"Optuna trials budget (= NiOA function evaluations) : {N_TRIALS}")
print(f"Optimisation subset  : train={N_OPT_TRAIN:,}  val={N_OPT_VAL:,}")

---
## 1 · Define the Optuna Objective Function

In [ ]:
def optuna_objective(trial: optuna.Trial) -> float:
    """
    Optuna objective — mirrors the NiOA search space exactly.
    Returns minimum validation MSE; inf on failure.
    """
    params = {
        "lstm_layers"   : trial.suggest_int(   "lstm_layers",   2,    3          ),
        "units"         : trial.suggest_categorical("units",    [64, 128]         ),
        "dropout"       : trial.suggest_float(  "dropout",      0.3,  0.6        ),
        "optimizer"     : "adamw",
        "learning_rate" : trial.suggest_float(  "learning_rate",5e-5, 5e-4, log=True),
        "batch_size"    : 32,
    }

    model = None
    try:
        model = create_lstm_model(params, SEQ_LEN, NUM_FEATS)
        cbs   = [
            EarlyStopping(
                monitor              = "val_loss",
                patience             = OPT_PATIENCE,
                restore_best_weights = True,
                verbose              = 0,
            ),
            TimeLimitCallback(max_seconds=OPT_TIME_LIMIT),
        ]
        history = model.fit(
            X_tr_opt, y_tr_opt,
            validation_data = (X_va_opt, y_va_opt),
            epochs          = OPT_EPOCHS,
            batch_size      = params["batch_size"],
            callbacks       = cbs,
            verbose         = 0,
        )
        val_loss = float(np.min(history.history["val_loss"]))

    except Exception as exc:
        print(f"  [Optuna trial {trial.number}] failed — {exc}")
        val_loss = np.inf

    finally:
        if model is not None:
            del model
        K.clear_session()
        gc.collect()

    return val_loss

print("Optuna objective function defined.")

---
## 2 · Run Optuna Study

In [ ]:
study = optuna.create_study(
    direction = "minimize",
    sampler   = optuna.samplers.TPESampler(seed=RANDOM_SEED),
    study_name= f"DRNN_Optuna_k{HORIZON}",
)

print(f"Running {N_TRIALS} Optuna trials ...")
study.optimize(optuna_objective, n_trials=N_TRIALS, show_progress_bar=True)

best_optuna_params = study.best_params
best_optuna_params["optimizer"] = "adamw"
best_optuna_params["batch_size"] = 32

print(f"\nBest Optuna validation MSE : {study.best_value:.8f}")
print("Best hyperparameters :")
for k_name, v in best_optuna_params.items():
    print(f"  {k_name} : {v}")

---
## 3 · Final Training with Best Optuna Parameters

In [ ]:
K.clear_session(); gc.collect()

final_model = create_lstm_model(best_optuna_params, SEQ_LEN, NUM_FEATS)
final_model.summary()

BATCH = best_optuna_params["batch_size"]
train_ds = make_tf_dataset(X_train, y_train, SEQ_LEN, NUM_FEATS, BATCH)
val_ds   = make_tf_dataset(X_val,   y_val,   SEQ_LEN, NUM_FEATS, BATCH)

history_optuna = final_model.fit(
    train_ds,
    validation_data  = val_ds,
    epochs           = FINAL_EPOCHS,
    steps_per_epoch  = len(X_train) // BATCH,
    validation_steps = len(X_val)   // BATCH,
    callbacks        = [EarlyStopping(
        monitor="val_loss", patience=FINAL_PATIENCE,
        restore_best_weights=True, verbose=1
    )],
    verbose=1,
)
print("Final Optuna-DRNN training complete.")

---
## 4 · Evaluation and Artefact Saving

In [ ]:
y_pred_optuna = final_model.predict(X_test, verbose=0).flatten()

metrics_optuna = save_benchmark_results(
    model_name     = "drnn_optuna",
    horizon        = HORIZON,
    y_true         = y_test,
    y_pred         = y_pred_optuna,
    benchmark_root = BENCHMARK_DIR,
    extra_meta     = {
        "best_params"   : to_python_types(best_optuna_params),
        "best_val_mse"  : float(study.best_value),
        "n_trials"      : N_TRIALS,
        "sampler"       : "TPE",
    },
)
print(f"DRNN + Optuna  →  {metrics_optuna}")

# Save model and scaler alongside the result
optuna_model_dir = os.path.join(BENCHMARK_DIR, f"horizon_{HORIZON}", "drnn_optuna")
os.makedirs(optuna_model_dir, exist_ok=True)
final_model.save(os.path.join(optuna_model_dir, f"drnn_optuna_k{HORIZON}.h5"))
joblib.dump(scaler, os.path.join(optuna_model_dir, "scaler.pkl"))

with open(os.path.join(optuna_model_dir, "best_params.json"), "w") as f:
    json.dump(to_python_types(best_optuna_params), f, indent=4)

print("Optuna-DRNN artefacts saved.")

---
## 5 · Plots

In [ ]:
plots_dir = os.path.join(optuna_model_dir, "plots")
os.makedirs(plots_dir, exist_ok=True)

# ── Optuna optimisation history ──────────────────────────────────────────────
trial_values = [t.value for t in study.trials if t.value is not None]
running_best = np.minimum.accumulate(trial_values)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(trial_values,  alpha=0.5, color="grey",       label="Trial val MSE")
ax.plot(running_best,  linewidth=2, color="darkorange", label="Running best")
ax.set_xlabel("Trial")
ax.set_ylabel("Validation MSE")
ax.set_title(f"Optuna Convergence — k = {HORIZON} s")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "optuna_convergence.png"), dpi=200)
plt.show(); plt.close()

# ── Training / validation loss ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_optuna.history["loss"],     label="Train loss")
ax.plot(history_optuna.history["val_loss"], label="Val loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE")
ax.set_title(f"DRNN+Optuna Training Curve — k = {HORIZON} s")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "training_curve.png"), dpi=200)
plt.show(); plt.close()

# ── Predicted vs Actual ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred_optuna, alpha=0.4, s=5, color="steelblue")
lim = [min(y_test.min(), y_pred_optuna.min()), max(y_test.max(), y_pred_optuna.max())]
ax.plot(lim, lim, "r--", linewidth=1.5)
ax.set_xlabel(f"Actual ΔE$_{{k={HORIZON}}}$ (kWh)")
ax.set_ylabel(f"Predicted ΔE$_{{k={HORIZON}}}$ (kWh)")
ax.set_title(f"DRNN+Optuna: Predicted vs Actual — k = {HORIZON} s")
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "pred_vs_actual.png"), dpi=200)
plt.show(); plt.close()

print(f"Plots saved to : {plots_dir}")